In [6]:
import random, statistics

class Card:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    def value(self):
        if self.rank in 'JQK': return 10  # Face cards = 10
        if self.rank == 'A':   return 11  # Ace = 11
        return int(self.rank)

    def __repr__(self): return f"{self.rank}{self.suit[0]}"


class Deck:
    SUITS = ['H','D','C','S']
    RANKS = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']

    def __init__(self, num_decks=6):
        self.num_decks    = num_decks
        self.needs_shuffle = False
        self._build()

    def _build(self):
        self.cards = [Card(r, s) for _ in range(self.num_decks)
                                 for s in self.SUITS for r in self.RANKS]
        self.shuffle()

    def shuffle(self):
        random.shuffle(self.cards)
        total = len(self.cards)
        # placing plastic card randomly in the last 10-25% of the shoe
        self.plastic       = random.randint(int(total * .75), int(total * .90))
        self.needs_shuffle = False

    def draw(self):
        if not self.cards: self._build()
        card = self.cards.pop(0)
        # flagging a reshuffle once we pass the plastic card position
        if len(self.cards) <= self.num_decks * 52 - self.plastic:
            self.needs_shuffle = True
        return card


# test
d = Deck(2)
print(f"Deck size: {len(d.cards)} (expect 104)")
print("5 cards:", [d.draw() for _ in range(5)])
print("Plastic position:", d.plastic)

Deck size: 104 (expect 104)
5 cards: [5C, 6H, KH, 9D, 10S]
Plastic position: 79


# Part 2 — UML Diagram

```
Card ◄── Deck                Hand
          │                   │
          └──── BlackjackGame ─┤
                    │         │
                 Player (base)─┘
                 ├── Dealer
                 ├── BasicPlayer
                 ├── HumanPlayer
                 ├── CountingPlayer      (Hi-Lo, Part 6)
                 └── BasicStrategyTablePlayer  (Part 10)
```
- Card: rank, suit, value()  
- Deck: multi-deck shoe, shuffle, draw, plastic card flag  
- Hand: list of cards, total value (soft/hard aces), bust/blackjack checks  
- Player: name, chips, hand, seen-cards list, decide() override  
- BlackjackGame: owns Deck + Dealer + players list; runs rounds and settles bets


In [7]:
# Part 3:

class Hand_sk:
    def add(self, card):          pass   
    def value(self):              pass   
    def is_bust(self):            pass  
    def is_blackjack(self):       pass   # 21 with exactly 2 cards

class Player_sk:
    def place_bet(self, amt):     pass   # deduct chips, store bet
    def see(self, card):          pass   # record a visible card
    def take(self, card):         pass   # add to hand + record
    def reset(self):              pass   # clear hand and bet
    def decide(self, up):         pass   # return 'hit' or 'stay'

class Game_sk:
    def _deal(self):              pass   # 2 cards to everyone
    def _play(self, player):      pass   # one player's hit/stay loop
    def _dealer_play(self):       pass   # dealer's fixed-rule loop
    def _settle(self, player):    pass   # compare hands, pay chips
    def play_round(self):         pass   # one full round
    def play_rounds(self, n):     pass   # n rounds

print("Skeleton defined.")
    

Skeleton defined.


In [8]:
# Part 4:
class Hand:
    def __init__(self): self.cards = []

    def add(self, c): self.cards.append(c)

    def value(self):
        total = sum(c.value() for c in self.cards)
        aces  = sum(1 for c in self.cards if c.rank == 'A')
        # Reduce each Ace from 11 to 1 while over 21
        while total > 21 and aces:
            total -= 10; aces -= 1
        return total

    def is_bust(self):      return self.value() > 21
    def is_blackjack(self): return len(self.cards) == 2 and self.value() == 21
    def __repr__(self):     return f"{self.cards}({self.value()})"


# Base Player
class Player:
    def __init__(self, name, chips=1000):
        self.name  = name
        self.chips = chips
        self.hand  = Hand()
        self.bet   = 0
        self.seen  = []   # every card this player has observed 

    def place_bet(self, amt):
        amt = min(amt, self.chips)
        self.chips -= amt; self.bet = amt

    def see(self, card):   self.seen.append(card)
    def take(self, card):  self.hand.add(card); self.see(card)   # add to hand + record it
    def reset(self):       self.hand = Hand(); self.bet = 0
    def decide(self, up, verbose=False): raise NotImplementedError
    def __repr__(self):    return f"{self.name}(${self.chips})"


# Concrete Players 
class Dealer(Player):
    """Hits on 16 or below, stays on 17+."""
    def __init__(self): super().__init__("Dealer", 999999)
    def decide(self, up=None, verbose=False):
        return 'hit' if self.hand.value() <= 16 else 'stay'


class BasicPlayer(Player):
    """Same fixed rule as the dealer."""
    def decide(self, up, verbose=False):
        return 'hit' if self.hand.value() <= 16 else 'stay'


class HumanPlayer(Player):
    """Asks the real user each turn."""
    def decide(self, up, verbose=False):
        print(f"Your hand: {self.hand}  Dealer shows: {up}")
        return 'hit' if input("(h)it or (s)tay? ")[0] == 'h' else 'stay'


# Hi-Lo card counting
class CountingPlayer(Player):
    """
    Hi-Lo count: 2-6 = +1, 7-9 = 0, 10-A = -1.
    Stay if running count > threshold, else hit (for hands 12-16).
    """
    def __init__(self, name, chips=1000, threshold=0):
        super().__init__(name, chips)
        self.threshold = threshold

    def count(self):
        c = 0
        for card in self.seen:
            if   card.rank in ['2','3','4','5','6']:   c += 1
            elif card.rank in ['10','J','Q','K','A']:  c -= 1
        return c

    def decide(self, up, verbose=False):
        hv = self.hand.value()
        if hv >= 17: return 'stay'   # hard rule: always stay
        if hv <= 11: return 'hit'    # hard rule: always hit
        # gray zone 12-16
        return 'stay' if self.count() > self.threshold else 'hit'


# canonical basic strategy table
class BasicStrategyTablePlayer(Player):
    """
    Hit/stay based on player total vs dealer up-card.
    13-16 vs weak dealer (2-6) -> stay (let dealer bust).
    """
    def decide(self, up, verbose=False):
        pv, dv = self.hand.value(), up.value()
        if pv <= 11:          return 'hit'
        if pv >= 17:          return 'stay'
        if pv == 12:          return 'stay' if 4 <= dv <= 6 else 'hit'
        if 13 <= pv <= 16:    return 'stay' if dv <= 6 else 'hit'
        return 'hit'


# Game 
class BlackjackGame:
    def __init__(self, players, num_decks=6, bet=10, verbose=False):
        self.deck    = Deck(num_decks)
        self.dealer  = Dealer()
        self.players = players
        self.bet     = bet
        self.v       = verbose

    def _broadcast(self, card, exclude=None):
        """All players except one observe a card (public info)."""
        for p in self.players:
            if p is not exclude: p.see(card)

    def _deal(self):
        for _ in range(2):
            for p in self.players:
                c = self.deck.draw(); p.take(c); self._broadcast(c, exclude=p)
            c = self.deck.draw(); self.dealer.take(c); self._broadcast(c)

    def _play(self, player):
        up = self.dealer.hand.cards[0]
        while not player.hand.is_bust():
            if player.decide(up, self.v) == 'hit':
                c = self.deck.draw(); player.take(c); self._broadcast(c, exclude=player)
                if self.v: print(f"  {player.name} hits -> {player.hand}")
            else:
                if self.v: print(f"  {player.name} stays at {player.hand.value()}")
                break

    def _dealer_play(self):
        if self.v: print(f"  Dealer: {self.dealer.hand}")
        while not self.dealer.hand.is_bust():
            if self.dealer.decide() == 'hit':
                c = self.deck.draw(); self.dealer.take(c); self._broadcast(c)
                if self.v: print(f"  Dealer hits -> {self.dealer.hand}")
            else:
                if self.v: print(f"  Dealer stays at {self.dealer.hand.value()}")
                break

    def _settle(self, p):
        pv, dv = p.hand.value(), self.dealer.hand.value()
        if   p.hand.is_bust():                                         out = 'lose'
        elif p.hand.is_blackjack() and not self.dealer.hand.is_blackjack(): out = 'blackjack'
        elif self.dealer.hand.is_bust() or pv > dv:                    out = 'win'
        elif pv == dv:                                                 out = 'push'
        else:                                                          out = 'lose'

        if   out == 'win':       p.chips += p.bet * 2
        elif out == 'blackjack': p.chips += int(p.bet * 2.5)
        elif out == 'push':      p.chips += p.bet
        if self.v: print(f"  {p.name}: {out} -> ${p.chips}")
        return out

    def play_round(self):
        for p in self.players: p.reset(); p.place_bet(self.bet)
        self.dealer.reset()
        if self.v: print("\n--- New Round ---")
        self._deal()
        for p in self.players:
            if self.v: print(f"\n{p.name}: {p.hand}")
            self._play(p)
        if self.v: print("\nDealer:")
        self._dealer_play()
        results = {p.name: self._settle(p) for p in self.players}
        if self.deck.needs_shuffle:
            self.deck.shuffle()
            if self.v: print("  [Shuffling deck]")
        return results

    def play_rounds(self, n):
        history = []
        for _ in range(n):
            self.players = [p for p in self.players if p.chips >= self.bet]
            if not self.players: break
            history.append(self.play_round())
        return history


print("All classes ready.")

All classes ready.


In [9]:
# Helper function
def run_game(threshold=0, rounds=50, chips=1000):
    """One game: CountingPlayer vs 3 BasicPlayers."""
    counter = CountingPlayer("Counter", chips, threshold)
    others  = [BasicPlayer(f"P{i}", chips) for i in range(3)]
    BlackjackGame([counter] + others).play_rounds(rounds)
    return counter.chips

def run_bs(rounds=50, chips=1000):
    """One game: BasicStrategyTablePlayer vs 3 BasicPlayers."""
    p      = BasicStrategyTablePlayer("BS", chips)
    others = [BasicPlayer(f"P{i}", chips) for i in range(3)]
    BlackjackGame([p] + others).play_rounds(rounds)
    return p.chips

def stats(results, start=1000):
    """Returns (mean, stdev, win_probability) for a list of final chip counts."""
    net = [r - start for r in results]
    return statistics.mean(results), statistics.stdev(results), sum(1 for n in net if n > 0) / len(net)

print("Helpers defined.")

Helpers defined.


In [10]:
# Part 5: 5 rounds
p1 = BasicPlayer("Alice", 200)
p2 = BasicPlayer("Bob",   200)
BlackjackGame([p1, p2], verbose=True).play_rounds(5)
print(f"\nAlice: ${p1.chips}   Bob: ${p2.chips}")


--- New Round ---

Alice: [5S, 9S](14)
  Alice hits -> [5S, 9S, 5S](19)
  Alice stays at 19

Bob: [6D, KS](16)
  Bob hits -> [6D, KS, 4H](20)
  Bob stays at 20

Dealer:
  Dealer: [JH, 2S](12)
  Dealer hits -> [JH, 2S, 4S](16)
  Dealer hits -> [JH, 2S, 4S, AD](17)
  Dealer stays at 17
  Alice: win -> $210
  Bob: win -> $210

--- New Round ---

Alice: [5D, JH](15)
  Alice hits -> [5D, JH, 7D](22)

Bob: [KD, 5H](15)
  Bob hits -> [KD, 5H, 8C](23)

Dealer:
  Dealer: [QC, JS](20)
  Dealer stays at 20
  Alice: lose -> $200
  Bob: lose -> $200

--- New Round ---

Alice: [3H, 10C](13)
  Alice hits -> [3H, 10C, 6D](19)
  Alice stays at 19

Bob: [10H, 10D](20)
  Bob stays at 20

Dealer:
  Dealer: [KC, 8H](18)
  Dealer stays at 18
  Alice: win -> $210
  Bob: win -> $210

--- New Round ---

Alice: [7C, 4D](11)
  Alice hits -> [7C, 4D, 6C](17)
  Alice stays at 17

Bob: [9C, 3H](12)
  Bob hits -> [9C, 3H, 7S](19)
  Bob stays at 19

Dealer:
  Dealer: [2D, 4S](6)
  Dealer hits -> [2D, 4S, 5D](11)
  D

In [11]:
# Part 7: single 50 rounds simulation
final = run_game(threshold=0, rounds=50)
print(f"Counter ended with ${final}   (net {final - 1000:+d})")

Counter ended with $925   (net -75)


In [12]:
# Part 8: 100 games x 50 rounds
w = [run_game() for _ in range(100)]
avg, std, wp = stats(w)
print(f"Avg chips: ${avg:.1f}   Net: {avg-1000:+.1f}   Std: ${std:.1f}   Win%: {wp:.0%}   Loss%: {1-wp:.0%}")

# Text histogram (no libraries needed)
print("\nHistogram of final chips:")
buckets = {}
for val in w:
    b = (val // 50) * 50
    buckets[b] = buckets.get(b, 0) + 1
for b in sorted(buckets):
    print(f"  ${b:>5}: {'#' * buckets[b]}")

Avg chips: $972.5   Net: -27.5   Std: $72.2   Win%: 32%   Loss%: 68%

Histogram of final chips:
  $  750: #
  $  800: ####
  $  850: ##########
  $  900: ####################
  $  950: ##############################
  $ 1000: ##################
  $ 1050: ##############
  $ 1100: ##
  $ 1150: #


In [13]:
# Part 9: Threshold scan
print(f"{'Threshold':>10}  {'Avg':>8}  {'Net':>7}  {'Std':>7}  {'Win%':>6}")
print("-" * 46)
for t in [-4, -2, 0, 2, 4]:
    res = [run_game(threshold=t) for _ in range(100)]
    avg, std, wp = stats(res)
    print(f"  {t:>+8}   ${avg:>6.0f}   {avg-1000:>+6.0f}   ${std:>5.0f}   {wp:>5.0%}")

 Threshold       Avg      Net      Std    Win%
----------------------------------------------
        -4   $   978      -22   $   77     39%
        -2   $   977      -23   $   68     39%
        +0   $   973      -27   $   73     27%
        +2   $   977      -23   $   80     37%
        +4   $   978      -22   $   71     30%


In [16]:
# Part 10: New strategy

bs_results = [run_bs()       for _ in range(100)]
ctr_results = [run_game()    for _ in range(100)]

ba, _, bwp = stats(bs_results)
ca, _, cwp = stats(ctr_results)

print(f"Basic Startegy Table : avg ${ba:.1f} net {ba-1000:+.1f} win% {bwp:.0%}")
print(f"Hi-Lo counting (t=0) : avg ${ca:.1f} net {ca-1000:+.1f} win% {cwp:.0%}")
print(f"\nBetter startegy: {'Basic Stategy Table' if ba < ca else 'Hi-Lo Counting'}")


Basic Startegy Table : avg $987.8 net -12.2 win% 43%
Hi-Lo counting (t=0) : avg $980.5 net -19.5 win% 41%

Better startegy: Hi-Lo Counting
